## `Modelos ligeros y rápidos`

Más de **20 modelos de lenguaje** que puedes usar para fine-tuning o generación, organizados por tipo y utilidad. Todos están disponibles en [Hugging Face](https://huggingface.co/models) y compatibles con `AutoModelForCausalLM` o variantes según el caso:

| Modelo | Descripción |
|--------|-------------|
| `gpt2` | Modelo clásico de generación de texto, rápido y versátil. |
| `distilgpt2` | Versión reducida de GPT-2, más eficiente. |
| `facebook/opt-125m` | Modelo causal de Meta, ideal para fine-tuning ligero. |
| `EleutherAI/gpt-neo-125M` | Alternativa abierta a GPT-2 con arquitectura moderna. |
| `tiiuae/falcon-1b` | Modelo pequeño de Falcon, eficiente y preciso. |
| `mistralai/Mistral-7B-v0.1` | Modelo compacto y potente, ideal para tareas técnicas. |
| `openlm-research/open_llama_3b` | Llama 3B abierto, útil para generación estructurada. |
| `google/flan-t5-small` | Modelo encoder-decoder para tareas de texto estructurado. |
| `Salesforce/codegen-350M-mono` | Optimizado para generación de código y texto técnico. |
| `mosaicml/mpt-1b-redpajama-200b` | Modelo causal entrenado con corpus técnico. |



## Modelos medianos para tareas más complejas

| Modelo | Descripción |
|--------|-------------|
| `gpt2-medium` | Versión intermedia de GPT-2, más capacidad. |
| `facebook/opt-350m` | Más potente que el 125M, útil para forecasting contextual. |
| `EleutherAI/gpt-neo-1.3B` | Modelo robusto para generación técnica y simulación. |
| `tiiuae/falcon-7b` | Modelo de alto rendimiento, ideal para simulaciones. |
| `mistralai/Mixtral-8x7B-Instruct-v0.1` | Mezcla de expertos, útil para tareas multivariadas. |
| `google/flan-t5-base` | Encoder-decoder para tareas de clasificación y generación. |
| `bigscience/bloom-560m` | Modelo multilingüe, útil para generación editorial. |
| `mosaicml/mpt-7b-storywriter` | Optimizado para generación secuencial y narrativa técnica. |

---

## Modelos grandes (requieren GPU potente)

| Modelo | Descripción |
|--------|-------------|
| `gpt2-large` | Versión avanzada de GPT-2, buena para forecasting detallado. |
| `facebook/opt-1.3b` | Modelo robusto para simulaciones y generación por series. |
| `EleutherAI/gpt-j-6B` | Potente modelo para generación técnica y contextual. |
| `bigscience/bloom-1b7` | Multilingüe y versátil, útil para flujos editoriales. |
| `meta-llama/Llama-2-7b-hf` | Modelo de Meta para tareas complejas y simulación. |



## ¿Cómo elegir?

- Para pruebas rápidas: `opt-125m`, `distilgpt2`, `gpt2`, `falcon-1b`
- Para forecasting contextual: `opt-350m`, `gpt-neo-1.3B`, `flan-t5-base`
- Para generación técnica editorial: `mpt-7b-storywriter`, `codegen`, `bloom`
- Para simulación multivariada: `Mixtral`, `Llama`, `Falcon-7b`



Dataset sintéticopara chatbots bancarios

- query_id: ID único de la consulta
- user_id: ID del usuario (para simular repetidos)
- query_text: Texto de la consulta (generado con patrones por intención)
- intention: Etiqueta de intención ML (e.g., 'check_balance', 'report_fraud', 'transfer_money', 'open_account', 'loan_inquiry', 'card_activation', 'payment_issue')
- timestamp: Fecha/hora de la consulta (estacionalidad: más en días laborables)
- account_type: Tipo de cuenta ('savings', 'checking', 'credit')
- user_age: Edad del usuario (18-80)
- user_location: Ubicación ('urban', 'rural')
- device: Dispositivo usado ('mobile', 'web', 'app')
- response_time: Tiempo de respuesta simulado (segundos, para métricas)
- satisfaction_score: Puntuación de satisfacción aleatoria (1-5)

Esto permite fine-tuning de un modelo de clasificación de intenciones con datos realistas.


In [ ]:
import pandas as pd  # Para manipulación de datos, creación de DataFrames y manejo de grandes datasets
import numpy as np   # Para generación de datos sintéticos numéricos, fechas aleatorias y operaciones matemáticas

import matplotlib.pyplot as plt  # Para visualizaciones básicas como gráficos de series temporales y forecasts
import seaborn as sns  # Para gráficos avanzados en EDA, como heatmaps o distribuciones

from datetime import datetime, timedelta  # Para generar timestamps realistas en consultas
import random  # Para selección aleatoria de categorías e intenciones

from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.model_selection import train_test_split  # Para dividir el dataset en train/test para fine-tuning
from sklearn.metrics import accuracy_score, classification_report  # Para evaluar el modelo de clasificación de intenciones
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments  # Hugging Face para tokenizer, modelo pre-entrenado, fine-tuning y entrenamiento
from transformers import TextClassificationPipeline  # Para pipeline de inferencia simple en el chatbot
from datasets import Dataset  # Para convertir DataFrame a formato Hugging Face Dataset para fine-tuning
import torch  # Backend para Hugging Face (verifica GPU si disponible)
import warnings
warnings.filterwarnings('ignore')  # Ignorar warnings menores para salida limpia

In [ ]:
def generate_bank_chat_data(num_records=100000):
    """
    Genera dataset sintético de consultas bancarias para chatbot.
    - num_records: Número de consultas (100,000).
    Retorna DataFrame con variables realistas para entrenamiento de intenciones.
    """
    np.random.seed(42)  # Reproducibilidad
    random.seed(42)
    
    data = {}
    
    # ID único
    data['query_id'] = np.arange(1, num_records + 1)
    
    # ID de usuario: 20,000 usuarios repetidos
    data['user_id'] = np.random.randint(1, 20001, num_records)
    
    # Timestamp: Fechas aleatorias en 2 años, más consultas en laborables (lunes-viernes)
    start_date = datetime(2022, 1, 1)
    dates = [start_date + timedelta(days=random.randint(0, 730)) for _ in range(num_records)]
    data['timestamp'] = [d + timedelta(hours=random.randint(8, 20)) if d.weekday() < 5 else d + timedelta(hours=random.randint(10, 18)) for d in dates]
    
    # Edad usuario: 18-80
    data['user_age'] = np.random.randint(18, 81, num_records)
    
    # Ubicación: Mayor urbano
    data['user_location'] = np.random.choice(['urban', 'rural'], num_records, p=[0.7, 0.3])
    
    # Dispositivo: Mayor mobile
    data['device'] = np.random.choice(['mobile', 'web', 'app'], num_records, p=[0.5, 0.3, 0.2])
    
    # Tipo de cuenta
    data['account_type'] = np.random.choice(['savings', 'checking', 'credit'], num_records)
    
    # Intención: 7 categorías balanceadas
    intentions = ['check_balance', 'report_fraud', 'transfer_money', 'open_account', 'loan_inquiry', 'card_activation', 'payment_issue']
    data['intention'] = np.random.choice(intentions, num_records)
    
    # Texto de consulta: Templates por intención + variabilidad
    templates = {
        'check_balance': ["What's my current balance?", "Check my account balance please.", "How much money do I have left?"],
        'report_fraud': ["I think my card was stolen.", "Report fraudulent transaction.", "Unauthorized charge on my account."],
        'transfer_money': ["Transfer $100 to my savings.", "Send money to friend.", "Make a wire transfer."],
        'open_account': ["How to open a new account?", "Apply for checking account.", "Open savings account online."],
        'loan_inquiry': ["What are loan rates?", "Apply for personal loan.", "Check eligibility for mortgage."],
        'card_activation': ["Activate my new credit card.", "How to activate debit card?", "Card activation code."],
        'payment_issue': ["Payment not going through.", "Bill payment error.", "Why was my payment declined?"]
    }
    
    # Generar queries con variabilidad (agregar ruido como números aleatorios)
    queries = []
    for intn in data['intention']:
        base = random.choice(templates[intn])
        noise = f" {random.randint(10, 1000)}" if random.random() > 0.5 else ""
        queries.append(base + noise)
    data['query_text'] = queries
    
    # Tiempo de respuesta: Aleatorio 10-60 seg
    data['response_time'] = np.random.randint(10, 61, num_records)
    
    # Puntuación de satisfacción: 1-5, mayor si respuesta rápida
    data['satisfaction_score'] = np.clip(5 - (data['response_time'] // 10), 1, 5) + np.random.choice([-1, 0, 1], num_records)
    data['satisfaction_score'] = np.clip(data['satisfaction_score'], 1, 5)
    
    df = pd.DataFrame(data)
    print(f"Dataset generado: {len(df)} registros con {len(df.columns)} variables.")
    return df

# Generar dataset
chat_df = generate_bank_chat_data(num_records=100000)

In [ ]:
# Guardar como CSV (opcional para reutilizar)
chat_df.to_csv('bank_chat_queries.csv', index=False)

In [ ]:
# Paso 2: Análisis Exploratorio de Datos (EDA)
# Resumen estadístico para entender distribuciones
print("\nEstadísticas descriptivas:")
print(chat_df.describe())

In [ ]:
# Distribución de intenciones
plt.figure(figsize=(10, 6))
sns.countplot(y='intention', data=chat_df, palette='viridis')
plt.title('Distribución de Intenciones en Consultas')
plt.xlabel('Conteo')
plt.show()

In [ ]:
# Longitud de queries por intención (para ver variabilidad)
chat_df['query_length'] = chat_df['query_text'].apply(len)
plt.figure(figsize=(12, 6))
sns.boxplot(x='intention', y='query_length', data=chat_df)
plt.title('Longitud de Consultas por Intención')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Paso 3: Preprocesamiento para Fine-Tuning con Hugging Face
# Dividir en train/test (80/20)
train_df, test_df = train_test_split(chat_df, test_size=0.2, stratify=chat_df['intention'], random_state=42)

In [ ]:
# Codificar labels (intenciones a números para clasificación)
label_encoder = LabelEncoder()
train_df['label'] = label_encoder.fit_transform(train_df['intention'])
test_df['label'] = label_encoder.transform(test_df['intention'])

In [ ]:
# Convertir a Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df[['query_text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['query_text', 'label']])

In [ ]:
# Cargar tokenizer y modelo base (distilbert para eficiencia, fine-tune para clasificación)
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_encoder.classes_))

In [ ]:
# Tokenizar datasets
def tokenize_function(examples):
    return tokenizer(examples['query_text'], padding='max_length', truncation=True, max_length=64)

In [ ]:
train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

In [ ]:
# Set format for PyTorch
train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

In [ ]:
from transformers import EarlyStoppingCallback  # Agregar para load_best_model_at_end

# Paso 4: Fine-Tuning del Modelo con Trainer
# Configurar argumentos de entrenamiento (epochs bajos para demo, ajusta para precisión)
training_args = TrainingArguments(
    output_dir='./bank_intent_model',
    num_train_epochs=3,  # 3 epochs para fine-tuning rápido
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy='epoch',  # Cambiado de 'evaluation_strategy' (deprecated en versiones nuevas)
    save_strategy='epoch',  # Añadido para coincidir con eval_strategy y evitar error con load_best_model_at_end
    logging_dir='./logs',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy'
)

# Función para métricas
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc}

# Trainer para fine-tuning (agregar callback para early stopping si load_best_model_at_end=True)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  # Agregar para load_best_model_at_end
)

# Entrenar modelo
print("Iniciando fine-tuning del modelo...")
trainer.train()

In [ ]:
# Guardar modelo fine-tuned
trainer.save_model('./bank_intent_model_finetuned')
tokenizer.save_pretrained('./bank_intent_model_finetuned')
print("Modelo fine-tuned guardado.")

In [ ]:
# Paso 5: Evaluación del Modelo
# Predicciones en test
predictions = trainer.predict(test_tokenized)
preds = np.argmax(predictions.predictions, axis=-1)
print("\nReporte de Clasificación:")
print(classification_report(test_df['label'], preds, target_names=label_encoder.classes_))

In [ ]:
from sklearn.metrics import confusion_matrix
# Matriz de confusión
cm = confusion_matrix(test_df['label'], preds)
plt.figure(figsize=(18, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Matriz de Confusión para Clasificación de Intenciones')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.show()

In [ ]:
# Paso 6: Implementar el Chatbot Simple con el Modelo Fine-Tuned
# Usamos pipeline para inferencia rápida
classifier = TextClassificationPipeline(model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Respuestas pre-definidas por intención (simulado chatbot)
responses = {
    'check_balance': "Su saldo actual es $1,234.56. ¿Necesita más detalles?",
    'report_fraud': "Hemos registrado su reporte de fraude. Un agente se contactará pronto.",
    'transfer_money': "Transferencia realizada exitosamente. Confirme el monto y destinatario.",
    'open_account': "Para abrir una cuenta, visite nuestra sucursal o app. ¿Qué tipo de cuenta desea?",
    'loan_inquiry': "Nuestras tasas de préstamo empiezan en 5%. ¿Cuánto necesita?",
    'card_activation': "Su tarjeta ha sido activada. Use PIN 1234 para primera transacción.",
    'payment_issue': "El pago falló por fondos insuficientes. Intente de nuevo o contacte soporte."
}

# Función para chatbot (loop simple en consola para demo; en producción, integra con web/app)
def run_chatbot():
    print("\n--- Chatbot Bancario ---")
    print("Escriba su consulta (o 'salir' para terminar):")
    while True:
        user_input = input("> ")
        if user_input.lower() == 'salir':
            break
        result = classifier(user_input)[0]
        intention = result['label'].split('LABEL_')[1]  # Extraer label (0-6) y map a intención
        intention_label = label_encoder.neceinverse_transform([int(intention)])[0]
        response = responses.get(intention_label, "Lo siento, no entendí su consulta. ¿Puede reformular?")
        print(f"Bot: {response}")

# Ejecutar chatbot demo
if __name__ == "__main__":
    run_chatbot()

print("\n¡Ejemplo end-to-end completado! El chatbot usa ML para detectar intenciones y responder acorde.")
print("- Dataset: 100k queries sintéticas con variabilidad para fine-tuning realista.")
print("- Modelo: Fine-tuned DistilBERT para clasificación de intenciones (eficiente y preciso).")
print("- Chatbot: Loop simple; en producción, integra con Rasa para conversaciones multi-turn o Flask para web.")
print("- Recomendación: Para escalabilidad, usa Hugging Face Hub para hostear modelo; agrega RAG para respuestas dinámicas con datos usuario.")